In [ ]:
# Goal: To perform an Independent Two-Sample t-test.

# This script reads an Excel file, splits scores by gender, checks whether the group variances are equal (F-test), and then runs the appropriate independent two-sample t-test:

# Welch's t-test if variances are not equal
# Pooled (equal-variance) t-test if variances are equal
# Input file used: independent-ttest-golfscores copy.xlsx

In [1]:
# --- Step 1: Import libraries (packages) ---
# pandas: to read Excel and work with tabular data
# numpy: to do numerical calculations
# scipy.stats: to run statistical tests (F distribution and t-test)

import pandas as pd
import numpy as np
from scipy import stats
from google.colab import files


In [4]:
# Step 2.1 — Upload the Excel input file


uploaded = files.upload()
input_filename = list(uploaded.keys())[0]
print("Uploaded file:", input_filename)

Saving independent-ttest-golfscores copy.xlsx to independent-ttest-golfscores copy.xlsx
Uploaded file: independent-ttest-golfscores copy.xlsx


In [5]:
# --- Step 2.2: Read the Excel file into a DataFrame ---

# read_excel() loads the Excel sheet into a table-like object called a DataFrame.
data = pd.read_excel(input_filename)

# Let's look at the first few rows to confirm it loaded correctly.
data


,male_score,female_score
0,82,75
1,80,76
2,85,80
3,85,77
4,78,80
5,87,77
6,82,73


In [6]:
# --- Step 3: Extract the columns of interest ---

# In pandas, we select columns using data["ColumnName"].

scores_male = data["male_score"]
scores_female  = data["female_score"]


In [7]:
# --- Step 4: Split the scores by gender ---


# Convert to numpy arrays (helps some stats functions)
scores_male = scores_male.dropna().to_numpy()
scores_female = scores_female.dropna().to_numpy()

# Print the sample sizes (how many observations in each group)
print("Number of male scores   =", len(scores_male))
print("Number of female scores =", len(scores_female))


Number of male scores   = 7
Number of female scores = 7


In [8]:
# Step 5: Check if both groups have enough observations because variance needs at least 2 data points.

if len(scores_male) < 2 or len(scores_female) < 2:
    print("Error: Not enough observations in one or both groups.")


In [10]:
# Step 6: F-test for equality of variances

# Only run the tests if both groups are large enough
if len(scores_male) >= 2 and len(scores_female) >= 2:

    # Sample variances (ddof=1 gives the sample variance)
    var_male = np.var(scores_male, ddof=1)
    var_female = np.var(scores_female, ddof=1)

    print("Sample variance (male)   =", var_male)
    print("Sample variance (female) =", var_female)

    # The F statistic is a ratio of variances.
    # To keep the F statistic >= 1, put the larger variance on top.
    if var_male >= var_female:
        f_stat = var_male / var_female
        dfn = len(scores_male) - 1  # numerator degrees of freedom
        dfd = len(scores_female) - 1  # denominator degrees of freedom
    else:
        f_stat = var_female / var_male
        dfn = len(scores_female) - 1
        dfd = len(scores_male) - 1

    # Two-sided p-value:
    #   p = 2 * min( P(F <= f_stat), P(F >= f_stat) )
    print("I am outside the if stmnt")
    cdf_value = stats.f.cdf(f_stat, dfn, dfd)
    print(cdf_value)
    p_value_ftest = 2 * min(cdf_value, 1 - cdf_value)

    # cdf refers to Cumulative Distribution Function

    print("\nF-test results (variance equality):")
    print("F statistic =", f_stat)
    print("df1 =", dfn, ", df2 =", dfd)
    print("p-value =", p_value_ftest)


Sample variance (male)   = 9.904761904761905
Sample variance (female) = 6.476190476190477
I am outside the if stmnt
0.6905656160170337

F-test results (variance equality):
F statistic = 1.5294117647058822
df1 = 6 , df2 = 6
p-value = 0.6188687679659326


In [11]:
# Step 7: Choose the correct independent t-test


# If p-value ≤ 0.05 in the F-test → assume variances are not equal → run Welch’s t-test
# If p-value > 0.05 → assume variances are equal → run the pooled (equal-variance) t-test

if len(scores_male) >= 2 and len(scores_female) >= 2:

    alpha = 0.05  # significance level

    if p_value_ftest <= alpha:
        print("\nVariances are NOT equal based on the F-test (p <= 0.05).")
        print("Running Welch's two-sample t-test (equal_var=False).")

        # Welch's t-test (does NOT assume equal variances)
        t_result = stats.ttest_ind(scores_male, scores_female, equal_var=False)
    else:
        print("\nVariances are equal based on the F-test (p > 0.05).")
        print("Running pooled (equal-variance) two-sample t-test (equal_var=True).")

        # Standard independent two-sample t-test (assumes equal variances)
        t_result = stats.ttest_ind(scores_male, scores_female, equal_var=True)

    print("\nT-test results:")
    print("t statistic =", t_result.statistic)
    print("p-value     =", t_result.pvalue)

    # Note:
    # If you switch the order of the groups (female first, then male),
    # the t statistic will flip sign, but the p-value will be the same.



Variances are equal based on the F-test (p > 0.05).
Running pooled (equal-variance) two-sample t-test (equal_var=True).

T-test results:
t statistic = 3.8288227591428385
p-value     = 0.0024006953026618435
